# 03 — Classical ML Models (v3) — 8-Model Optimized Training
## SOH Regression: Cross-Battery Generalization Split

**v3 Pipeline (bug fixes over v2):**
- Load preprocessed `battery_features.csv` from NB02 (18 features)
- **Cross-battery grouped split** (v2 bug: used intra-battery 80/20 → data leakage)
- Train 8 core models with 18 features (v2 had 12)
- Proper NaN imputation (no more `fillna(0)` for Re/Rct)
- Target: ≥95% within-±5% SOH accuracy on all models
- Save artifacts to `artifacts/v3/`

**v3 Bug Fixes:**
1. Split: Intra-battery → cross-battery (no leakage)
2. Features: 12 → 18 (6 new physics-informed features)
3. Imputation: `fillna(0)` → ffill/bfill/median (already done in NB02)
4. Scaler: single consistent scaler from NB02 training split

**Models (8 total):**
1. ExtraTrees (tree-based, unscaled)
2. GradientBoosting (sequential ensemble)
3. RandomForest (bagging ensemble)
4. XGBoost (boosted trees with tuning)
5. LightGBM (fast gradient boosting)
6. SVR (support vector regression)
7. Ridge (linear with L2)
8. KNN-5 (instance-based with distance weighting)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from src.utils.config import get_version_paths, ensure_version_dirs, FEATURE_COLS_V3

print('Setup complete.')

Setup complete.


In [2]:
# Setup v3 paths
v3 = get_version_paths('v3')
ensure_version_dirs('v3')

V3_FEATURES = v3['root'] / 'features'

print(f'v3 Results:  {v3["results"]}')
print(f'v3 Models:   {v3["models_classical"]}')
print(f'v3 Scalers:  {v3["scalers"]}')
print(f'v3 Features: {V3_FEATURES}')

v3 Results:  E:\VIT\aiBatteryLifecycle\artifacts\v3\results
v3 Models:   E:\VIT\aiBatteryLifecycle\artifacts\v3\models\classical
v3 Scalers:  E:\VIT\aiBatteryLifecycle\artifacts\v3\scalers
v3 Features: E:\VIT\aiBatteryLifecycle\artifacts\v3\features


In [3]:
# Load preprocessed features from NB02 (v3: 18 features, already imputed)
features_df = pd.read_csv(V3_FEATURES / 'battery_features.csv')
print(f'Dataset shape: {features_df.shape}')
print(f'Batteries: {sorted(features_df["battery_id"].unique())}')
print(f'SOH range: {features_df["SoH"].min():.1f}% — {features_df["SoH"].max():.1f}%')
print(f'NaN count: {features_df[FEATURE_COLS_V3].isna().sum().sum()}')

Dataset shape: (2678, 25)
Batteries: ['B0005', 'B0006', 'B0007', 'B0018', 'B0025', 'B0026', 'B0027', 'B0028', 'B0029', 'B0030', 'B0031', 'B0032', 'B0033', 'B0034', 'B0036', 'B0038', 'B0039', 'B0040', 'B0041', 'B0042', 'B0043', 'B0044', 'B0045', 'B0046', 'B0047', 'B0048', 'B0053', 'B0054', 'B0055', 'B0056']
SOH range: 2.2% — 122.2%
NaN count: 0


In [4]:
# ── v3 FIX: Cross-battery grouped split (no data leakage) ──
# v2 bug: intra-battery 80/20 chronological split per battery
#   → All batteries appear in both train AND test → inflated R²
# v3 fix: entire batteries in train OR test, never both

from src.data.preprocessing import group_battery_split

train_df, test_df = group_battery_split(features_df, train_ratio=0.8)

print(f'Train: {len(train_df)} samples from {train_df["battery_id"].nunique()} batteries')
print(f'Test:  {len(test_df)} samples from {test_df["battery_id"].nunique()} batteries')
print(f'Train batteries: {sorted(train_df["battery_id"].unique())}')
print(f'Test batteries:  {sorted(test_df["battery_id"].unique())}')

overlap = set(train_df['battery_id']) & set(test_df['battery_id'])
print(f'Overlap: {overlap if overlap else "NONE ✓ (no leakage)"}')
print(f'Train SOH: {train_df["SoH"].min():.1f}% — {train_df["SoH"].max():.1f}%')
print(f'Test SOH:  {test_df["SoH"].min():.1f}% — {test_df["SoH"].max():.1f}%')

Train: 2163 samples from 24 batteries
Test:  515 samples from 6 batteries
Train batteries: ['B0005', 'B0006', 'B0007', 'B0018', 'B0025', 'B0026', 'B0029', 'B0030', 'B0032', 'B0033', 'B0034', 'B0038', 'B0039', 'B0040', 'B0041', 'B0044', 'B0045', 'B0046', 'B0047', 'B0048', 'B0053', 'B0054', 'B0055', 'B0056']
Test batteries:  ['B0027', 'B0028', 'B0031', 'B0036', 'B0042', 'B0043']
Overlap: NONE ✓ (no leakage)
Train SOH: 2.2% — 101.8%
Test SOH:  2.8% — 122.2%


In [5]:
# v3: Use all 18 features (12 base + 6 physics-informed)
feature_cols = [c for c in FEATURE_COLS_V3 if c in features_df.columns]
print(f'Using {len(feature_cols)} features: {feature_cols}')

X_train = train_df[feature_cols].values
y_train = train_df['SoH'].values
X_test = test_df[feature_cols].values
y_test = test_df['SoH'].values

print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'y_test:  {y_test.shape}')

Using 18 features: ['cycle_number', 'ambient_temperature', 'peak_voltage', 'min_voltage', 'voltage_range', 'avg_current', 'avg_temp', 'temp_rise', 'cycle_duration', 'Re', 'Rct', 'delta_capacity', 'capacity_retention', 'cumulative_energy', 'dRe_dn', 'dRct_dn', 'soh_rolling_mean', 'voltage_slope']
X_train: (2163, 18)
y_train: (2163,)
X_test:  (515, 18)
y_test:  (515,)


In [6]:
# Load scaler from NB02 (v3: consistent with training split)
scaler = joblib.load(v3['scalers'] / 'v3_features_standard.joblib')
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Scaler loaded from NB02 (fitted on training batteries only).')
print(f'  Mean range: [{scaler.mean_.min():.4f}, {scaler.mean_.max():.4f}]')

Scaler loaded from NB02 (fitted on training batteries only).
  Mean range: [-0.0001, 3282.9991]


In [7]:
def evaluate_model(model_name, model, X_eval, y_eval, save_path=None):
    """Evaluate model and optionally save it."""
    y_pred = model.predict(X_eval)
    
    r2 = r2_score(y_eval, y_pred)
    mae = mean_absolute_error(y_eval, y_pred)
    within_5pct = float((np.abs(y_pred - y_eval) <= 5).mean() * 100)
    
    status = '✓ PASS' if within_5pct >= 95.0 else '✗ FAIL'
    print(f'{model_name:20s} | R²={r2:.4f} | MAE={mae:.2f} | Within-5%={within_5pct:.1f}% | {status}')
    
    if save_path:
        joblib.dump(model, save_path)
    
    return y_pred, {'model': model_name, 'r2': r2, 'mae': mae, 'within_5pct': within_5pct}

In [8]:
# ExtraTrees (unscaled)
model_et = ExtraTreesRegressor(
    n_estimators=1000,
    min_samples_leaf=2,
    max_features=0.7,
    random_state=42,
    n_jobs=-1
)
model_et.fit(X_train, y_train)
_, metrics_et = evaluate_model('ExtraTrees', model_et, X_test, y_test,
                               v3['models_classical'] / 'extra_trees.joblib')

ExtraTrees           | R²=0.9701 | MAE=3.20 | Within-5%=75.1% | ✗ FAIL


In [9]:
# GradientBoosting (unscaled)
model_gb = GradientBoostingRegressor(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
model_gb.fit(X_train, y_train)
_, metrics_gb = evaluate_model('GradientBoosting', model_gb, X_test, y_test,
                               v3['models_classical'] / 'gradient_boosting.joblib')

GradientBoosting     | R²=0.9860 | MAE=1.38 | Within-5%=95.1% | ✓ PASS


In [10]:
# RandomForest (unscaled)
model_rf = RandomForestRegressor(
    n_estimators=1000,
    min_samples_leaf=2,
    max_features=0.7,
    random_state=42,
    n_jobs=-1
)
model_rf.fit(X_train, y_train)
_, metrics_rf = evaluate_model('RandomForest', model_rf, X_test, y_test,
                               v3['models_classical'] / 'random_forest.joblib')

RandomForest         | R²=0.9814 | MAE=1.83 | Within-5%=91.3% | ✗ FAIL


In [11]:
# XGBoost (unscaled, tuned hyperparameters)
model_xgb = XGBRegressor(
    n_estimators=1200,
    max_depth=9,
    learning_rate=0.02,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
model_xgb.fit(X_train, y_train)
_, metrics_xgb = evaluate_model('XGBoost', model_xgb, X_test, y_test,
                                v3['models_classical'] / 'xgboost.joblib')

XGBoost              | R²=0.9866 | MAE=1.58 | Within-5%=93.8% | ✗ FAIL


In [12]:
# LightGBM (unscaled, tuned hyperparameters)
model_lgbm = LGBMRegressor(
    n_estimators=1200,
    num_leaves=127,
    learning_rate=0.02,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
model_lgbm.fit(X_train, y_train)
_, metrics_lgbm = evaluate_model('LightGBM', model_lgbm, X_test, y_test,
                                 v3['models_classical'] / 'lightgbm.joblib')

LightGBM             | R²=0.9826 | MAE=1.98 | Within-5%=89.5% | ✗ FAIL


In [13]:
# SVR (scaled)
model_svr = SVR(
    C=1000.0,
    epsilon=0.1,
    kernel='rbf'
)
model_svr.fit(X_train_scaled, y_train)
_, metrics_svr = evaluate_model('SVR', model_svr, X_test_scaled, y_test,
                               v3['models_classical'] / 'svr.joblib')

SVR                  | R²=0.8898 | MAE=4.92 | Within-5%=79.0% | ✗ FAIL


In [14]:
# Ridge (scaled)
model_ridge = Ridge(
    alpha=0.1
)
model_ridge.fit(X_train_scaled, y_train)
_, metrics_ridge = evaluate_model('Ridge', model_ridge, X_test_scaled, y_test,
                                  v3['models_classical'] / 'ridge.joblib')

Ridge                | R²=0.9656 | MAE=3.23 | Within-5%=88.9% | ✗ FAIL


In [15]:
# KNN-5 (scaled, with distance weighting)
model_knn5 = KNeighborsRegressor(
    n_neighbors=5,
    weights='distance',
    n_jobs=-1
)
model_knn5.fit(X_train_scaled, y_train)
_, metrics_knn5 = evaluate_model('KNN-5', model_knn5, X_test_scaled, y_test,
                                v3['models_classical'] / 'knn_k5.joblib')

KNN-5                | R²=0.7555 | MAE=11.02 | Within-5%=34.2% | ✗ FAIL


In [16]:
# Collect results
all_metrics = [
    metrics_et, metrics_gb, metrics_rf, metrics_xgb, metrics_lgbm,
    metrics_svr, metrics_ridge, metrics_knn5
]

results_df = pd.DataFrame(all_metrics)
results_df = results_df.sort_values('within_5pct', ascending=False)

print('\n' + '='*70)
print('FINAL RESULTS — v3 Classical ML (Cross-Battery Split, 18 Features)')
print('='*70)
print(results_df.to_string(index=False))

# Count passes
n_passed = (results_df['within_5pct'] >= 95.0).sum()
print(f'\nPassed (≥95%): {n_passed}/8')

# Save results (v3: consistent naming)
results_df.to_csv(v3['results'] / 'v3_classical_soh_results.csv', index=False)
print(f'\nResults saved to {v3["results"] / "v3_classical_soh_results.csv"}')


FINAL RESULTS — v3 Classical ML (Cross-Battery Split, 18 Features)
           model       r2       mae  within_5pct
GradientBoosting 0.985984  1.383230    95.145631
         XGBoost 0.986594  1.576671    93.786408
    RandomForest 0.981407  1.834184    91.262136
        LightGBM 0.982554  1.976782    89.514563
           Ridge 0.965638  3.225993    88.932039
             SVR 0.889759  4.923939    79.029126
      ExtraTrees 0.970125  3.201794    75.145631
           KNN-5 0.755476 11.023403    34.174757

Passed (≥95%): 1/8

Results saved to E:\VIT\aiBatteryLifecycle\artifacts\v3\results\v3_classical_soh_results.csv
